# Class 1 & 2: NLP and Search
## Learning Notebook Part 3 - Industry Tools: NLTK, spaCy & Hugging Face

**Welcome to Part 3!** Now that you've built everything from scratch, let's see how industry professionals do it!

**What you've learned so far:**
- Part 1: Text preprocessing, tokenization, Bag of Words (from scratch)
- Part 2: TF-IDF, similarity search, hybrid search (from scratch)

**Now in Part 3, you'll learn:**
- 🔧 **NLTK**: Natural Language Toolkit for preprocessing and tokenization
- ⚡ **spaCy**: Industrial-strength NLP library with built-in models
- 🤗 **Hugging Face**: Modern transformers and tokenizers
- 📚 **Documentation**: How to find and use official docs for these tools

**Why learn these tools?**
- **Faster development**: No need to implement everything from scratch
- **Production-ready**: Battle-tested, optimized, and maintained
- **Rich features**: Advanced NLP capabilities out of the box
- **Community support**: Large ecosystems and active communities

**💡 Important**: Understanding the fundamentals (Parts 1 & 2) helps you:
- Choose the right tool for the job
- Debug issues when things go wrong
- Customize tools for your specific needs
- Understand what's happening under the hood

Let's explore the tools of the trade! 🚀


## Setup & Installation

**Installation commands** (run in terminal/command prompt):

```bash
# NLTK
pip install nltk

# spaCy
pip install spacy
python -m spacy download en_core_web_sm  # Download English model

# Hugging Face Transformers
pip install transformers

# scikit-learn (if not already installed)
pip install scikit-learn
```

**📚 Documentation Links:**
- **NLTK**: https://www.nltk.org/
- **spaCy**: https://spacy.io/
- **Hugging Face**: https://huggingface.co/docs/transformers
- **scikit-learn**: https://scikit-learn.org/stable/


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# NLTK
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tag import pos_tag

# spaCy
import spacy

# Hugging Face
from transformers import AutoTokenizer

# scikit-learn (for TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ All libraries imported successfully!")
print("\n📚 Documentation:")
print("  - NLTK: https://www.nltk.org/")
print("  - spaCy: https://spacy.io/")
print("  - Hugging Face: https://huggingface.co/docs/transformers")


In [ ]:
# Download NLTK data (run once)
# Uncomment and run the first time:

# nltk.download('punkt')  # Tokenizers
# nltk.download('stopwords')  # Stop words
# nltk.download('wordnet')  # For lemmatization
# nltk.download('averaged_perceptron_tagger')  # For POS tagging

print("💡 Tip: Run nltk.download() commands above if you get errors about missing data")
print("📚 NLTK Data: https://www.nltk.org/data.html")


In [ ]:
# Load spaCy model
# Make sure you've downloaded it: python -m spacy download en_core_web_sm
try:
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy model loaded successfully!")
except OSError:
    print("⚠️  spaCy model not found. Run: python -m spacy download en_core_web_sm")
    print("📚 spaCy Models: https://spacy.io/models")
    nlp = None


In [ ]:
# Load movie data
import os

if not os.path.exists('data/movies.csv'):
    print("Data file not found. Downloading from GitHub...")
    os.makedirs('data', exist_ok=True)
    import urllib.request
    url = 'https://raw.githubusercontent.com/samsung-ai-course/8th-9th-edition/main/Chapter%202%20-%20Natural%20Language%20Processing/Class%201%20%26%202%20-%20NLP%20and%20Search/data/movies.csv'
    urllib.request.urlretrieve(url, 'data/movies.csv')
    print("✓ Data file downloaded successfully!")

df = pd.read_csv('data/movies.csv')
print(f"Loaded {len(df)} movies")
df.head()


---

## 1. Text Preprocessing with Industry Tools

**What we'll cover:**
- Tokenization (NLTK, spaCy, Hugging Face)
- Stop word removal
- Stemming and lemmatization
- Part-of-speech tagging

**📚 Documentation:**
- **NLTK Tokenization**: https://www.nltk.org/api/nltk.tokenize.html
- **spaCy Processing**: https://spacy.io/usage/linguistic-features
- **Hugging Face Tokenizers**: https://huggingface.co/docs/transformers/main_classes/tokenizer


### 1.1 Tokenization Comparison

Let's compare how different tools tokenize text:


In [ ]:
# Sample text
sample_text = "I don't think it's working. Let's try again!"
print(f"Original text: {sample_text}")
print("\n" + "=" * 70)

# Method 1: NLTK Tokenization
print("\n1. NLTK Tokenization:")
print("   Documentation: https://www.nltk.org/api/nltk.tokenize.html")
nltk_tokens = word_tokenize(sample_text)
print(f"   Tokens: {nltk_tokens}")
print(f"   Count: {len(nltk_tokens)} tokens")

# Method 2: spaCy Tokenization
print("\n2. spaCy Tokenization:")
print("   Documentation: https://spacy.io/usage/linguistic-features#tokenization")
if nlp:
    doc = nlp(sample_text)
    spacy_tokens = [token.text for token in doc]
    print(f"   Tokens: {spacy_tokens}")
    print(f"   Count: {len(spacy_tokens)} tokens")
else:
    print("   ⚠️  spaCy model not loaded")

# Method 3: Hugging Face Tokenization
print("\n3. Hugging Face Tokenization:")
print("   Documentation: https://huggingface.co/docs/transformers/main_classes/tokenizer")
try:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    hf_tokens = tokenizer.tokenize(sample_text)
    print(f"   Tokens: {hf_tokens}")
    print(f"   Count: {len(hf_tokens)} tokens")
    print("   Note: BERT uses WordPiece tokenization (subword tokens)")
except Exception as e:
    print(f"   ⚠️  Error: {e}")

print("\n💡 Key Differences:")
print("  - NLTK: Simple word tokenization, handles contractions")
print("  - spaCy: Smart tokenization, preserves linguistic structure")
print("  - Hugging Face: Subword tokenization (better for ML models)")


### 1.2 Stop Word Removal

**Stop words** are common words ("the", "is", "a") that are often filtered out.


In [ ]:
# Sample text
text = "The quick brown fox jumps over the lazy dog"
print(f"Original: {text}")
print("\n" + "=" * 70)

# Method 1: NLTK Stop Words
print("\n1. NLTK Stop Words:")
print("   Documentation: https://www.nltk.org/api/nltk.corpus.html#module-nltk.corpus")
nltk_stop_words = set(stopwords.words('english'))
nltk_tokens = word_tokenize(text.lower())
nltk_filtered = [word for word in nltk_tokens if word not in nltk_stop_words]
print(f"   Stop words: {sorted(nltk_stop_words)[:10]}... (total: {len(nltk_stop_words)})")
print(f"   Filtered: {nltk_filtered}")

# Method 2: spaCy Stop Words
print("\n2. spaCy Stop Words:")
print("   Documentation: https://spacy.io/api/stop-words")
if nlp:
    doc = nlp(text)
    spacy_filtered = [token.text for token in doc if not token.is_stop]
    spacy_stop_count = sum(1 for token in doc if token.is_stop)
    print(f"   Stop words detected: {spacy_stop_count}")
    print(f"   Filtered: {spacy_filtered}")
else:
    print("   ⚠️  spaCy model not loaded")

print("\n💡 Tip: Both libraries have built-in stop word lists. Choose based on your needs!")


### 1.3 Stemming vs Lemmatization

**Stemming**: Reduces words to their root form (faster, less accurate)
**Lemmatization**: Reduces words to their dictionary form (slower, more accurate)


In [ ]:
# Sample words
words = ["running", "runs", "ran", "better", "best", "studies", "studying"]
print("Original words:", words)
print("\n" + "=" * 70)

# Method 1: NLTK Stemming (Porter Stemmer)
print("\n1. NLTK Stemming (Porter Stemmer):")
print("   Documentation: https://www.nltk.org/api/nltk.stem.html")
stemmer = PorterStemmer()
nltk_stems = [stemmer.stem(word) for word in words]
print(f"   Stems: {nltk_stems}")

# Method 2: NLTK Lemmatization
print("\n2. NLTK Lemmatization:")
print("   Documentation: https://www.nltk.org/api/nltk.stem.wordnet.html")
lemmatizer = WordNetLemmatizer()
nltk_lemmas = [lemmatizer.lemmatize(word) for word in words]
print(f"   Lemmas: {nltk_lemmas}")
print("   Note: Lemmatization may need POS tags for better results")

# Method 3: spaCy Lemmatization
print("\n3. spaCy Lemmatization:")
print("   Documentation: https://spacy.io/api/lemmatizer")
if nlp:
    text = " ".join(words)
    doc = nlp(text)
    spacy_lemmas = [token.lemma_ for token in doc]
    print(f"   Lemmas: {spacy_lemmas}")
    print("   Note: spaCy automatically uses POS tags for better lemmatization")
else:
    print("   ⚠️  spaCy model not loaded")

print("\n💡 Comparison:")
print("  - Stemming: Fast, but can produce non-words (e.g., 'run' → 'run', 'better' → 'better')")
print("  - Lemmatization: Slower, but produces valid words (e.g., 'better' → 'good' with POS tag)")
print("  - spaCy: Best of both - fast AND accurate with built-in POS tagging")


### 1.4 Complete Preprocessing Pipeline

Let's build a complete preprocessing function using industry tools:


In [ ]:
def preprocess_nltk(text, remove_stopwords=True, lemmatize=True):
    """
    Preprocess text using NLTK.
    
    Documentation: https://www.nltk.org/
    """
    # Tokenize
    tokens = word_tokenize(text.lower())
    
    # Remove stop words
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [token for token in tokens if token not in stop_words]
    
    # Remove punctuation and keep only alphanumeric
    tokens = [token for token in tokens if token.isalnum()]
    
    # Lemmatize
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return tokens

def preprocess_spacy(text, remove_stopwords=True, lemmatize=True):
    """
    Preprocess text using spaCy.
    
    Documentation: https://spacy.io/usage/linguistic-features
    """
    if not nlp:
        return []
    
    doc = nlp(text.lower())
    
    # Extract tokens based on criteria
    tokens = []
    for token in doc:
        # Skip stop words if requested
        if remove_stopwords and token.is_stop:
            continue
        # Skip punctuation
        if token.is_punct:
            continue
        # Skip spaces
        if token.is_space:
            continue
        
        # Get lemma or text
        if lemmatize:
            tokens.append(token.lemma_)
        else:
            tokens.append(token.text)
    
    return tokens

# Test both methods
sample_text = "I'm running quickly through the beautiful garden!"
print(f"Original: {sample_text}")
print("\n" + "=" * 70)

print("\nNLTK Preprocessing:")
print("  Documentation: https://www.nltk.org/")
nltk_result = preprocess_nltk(sample_text)
print(f"  Result: {nltk_result}")

print("\nspaCy Preprocessing:")
print("  Documentation: https://spacy.io/usage/linguistic-features")
if nlp:
    spacy_result = preprocess_spacy(sample_text)
    print(f"  Result: {spacy_result}")
else:
    print("  ⚠️  spaCy model not loaded")

print("\n💡 Both methods achieve similar results, but spaCy is faster for large datasets!")


---

## 2. TF-IDF with scikit-learn (Advanced Features)

**What we'll cover:**
- Advanced TfidfVectorizer parameters
- Custom tokenizers and preprocessors
- Feature extraction and analysis

**📚 Documentation:**
- **TfidfVectorizer**: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html


In [ ]:
# Advanced TF-IDF with custom tokenizer
print("=" * 70)
print("Advanced TF-IDF with scikit-learn")
print("Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html")
print("=" * 70)

# Sample documents
documents = df['description'].head(10).tolist()

# Method 1: Basic TF-IDF (what we used before)
print("\n1. Basic TF-IDF:")
basic_vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
basic_vectors = basic_vectorizer.fit_transform(documents)
print(f"   Shape: {basic_vectors.shape}")
print(f"   Vocabulary size: {len(basic_vectorizer.get_feature_names_out())}")

# Method 2: TF-IDF with NLTK tokenizer
print("\n2. TF-IDF with NLTK Tokenizer:")
def nltk_tokenizer(text):
    tokens = word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token.isalnum() and token not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

nltk_vectorizer = TfidfVectorizer(
    tokenizer=nltk_tokenizer,
    max_features=50,
    lowercase=False  # We handle lowercase in tokenizer
)
nltk_vectors = nltk_vectorizer.fit_transform(documents)
print(f"   Shape: {nltk_vectors.shape}")
print(f"   Vocabulary: {list(nltk_vectorizer.get_feature_names_out())[:10]}...")

# Method 3: TF-IDF with spaCy tokenizer
print("\n3. TF-IDF with spaCy Tokenizer:")
if nlp:
    def spacy_tokenizer(text):
        doc = nlp(text.lower())
        return [token.lemma_ for token in doc 
                if not token.is_stop and not token.is_punct and not token.is_space]
    
    spacy_vectorizer = TfidfVectorizer(
        tokenizer=spacy_tokenizer,
        max_features=50,
        lowercase=False
    )
    spacy_vectors = spacy_vectorizer.fit_transform(documents)
    print(f"   Shape: {spacy_vectors.shape}")
    print(f"   Vocabulary: {list(spacy_vectorizer.get_feature_names_out())[:10]}...")
else:
    print("   ⚠️  spaCy model not loaded")

print("\n💡 Key Takeaway:")
print("  - scikit-learn's TfidfVectorizer is flexible - you can use any tokenizer!")
print("  - Combine the power of NLTK/spaCy preprocessing with scikit-learn's TF-IDF")
print("  - Documentation: https://scikit-learn.org/stable/modules/feature_extraction.html")


---

## 3. Similarity Search with Industry Tools

**What we'll cover:**
- Using scikit-learn for similarity search
- Comparing different preprocessing approaches
- Performance considerations

**📚 Documentation:**
- **cosine_similarity**: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html


In [ ]:
# Similarity Search with Industry Tools
print("=" * 70)
print("Similarity Search with scikit-learn")
print("Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html")
print("=" * 70)

# Create TF-IDF vectors with spaCy preprocessing
if nlp:
    def spacy_tokenizer(text):
        doc = nlp(text.lower())
        return [token.lemma_ for token in doc 
                if not token.is_stop and not token.is_punct and not token.is_space]
    
    vectorizer = TfidfVectorizer(
        tokenizer=spacy_tokenizer,
        max_features=100,
        lowercase=False
    )
    
    # Fit on all documents
    tfidf_vectors = vectorizer.fit_transform(df['description'])
    print(f"\n✓ Created TF-IDF matrix: {tfidf_vectors.shape}")
    
    # Test query
    query = "space adventure exploration"
    print(f"\nQuery: '{query}'")
    
    # Convert query to vector
    query_vector = vectorizer.transform([query])
    
    # Calculate similarities
    similarities = cosine_similarity(query_vector, tfidf_vectors)[0]
    
    # Get top 5 results
    top_indices = similarities.argsort()[-5:][::-1]
    
    print("\nTop 5 Results:")
    for i, idx in enumerate(top_indices, 1):
        print(f"\n{i}. {df.iloc[idx]['title']}")
        print(f"   Similarity: {similarities[idx]:.3f}")
        print(f"   Description: {df.iloc[idx]['description'][:100]}...")
    
    print("\n💡 This is the same approach from Part 2, but with better preprocessing!")
    print("   Documentation: https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity")
else:
    print("⚠️  spaCy model not loaded. Using basic TF-IDF instead.")
    vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
    tfidf_vectors = vectorizer.fit_transform(df['description'])
    print(f"✓ Created TF-IDF matrix: {tfidf_vectors.shape}")


---

## 4. Hugging Face Tokenizers (Preview)

**What we'll cover:**
- Hugging Face tokenizers for transformer models
- Subword tokenization (WordPiece, BPE)
- When to use Hugging Face tokenizers

**📚 Documentation:**
- **Hugging Face Tokenizers**: https://huggingface.co/docs/transformers/main_classes/tokenizer
- **Tokenizer Library**: https://huggingface.co/docs/tokenizers/


In [ ]:
# Hugging Face Tokenizers
print("=" * 70)
print("Hugging Face Tokenizers")
print("Documentation: https://huggingface.co/docs/transformers/main_classes/tokenizer")
print("=" * 70)

sample_text = "I don't think transformers are complicated!"
print(f"\nOriginal text: {sample_text}")

# Different tokenizers for different models
models_to_try = [
    ("bert-base-uncased", "BERT - WordPiece tokenization"),
    ("gpt2", "GPT-2 - Byte Pair Encoding (BPE)"),
    ("t5-small", "T5 - SentencePiece tokenization")
]

for model_name, description in models_to_try:
    try:
        print(f"\n{description}:")
        print(f"  Model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Tokenize
        tokens = tokenizer.tokenize(sample_text)
        print(f"  Tokens: {tokens}")
        print(f"  Count: {len(tokens)} tokens")
        
        # Convert to IDs
        token_ids = tokenizer.encode(sample_text)
        print(f"  Token IDs: {token_ids[:10]}... (showing first 10)")
        
    except Exception as e:
        print(f"  ⚠️  Error loading {model_name}: {e}")
        print(f"  💡 Tip: First time loading a model downloads it from Hugging Face Hub")

print("\n💡 Key Points:")
print("  - Hugging Face tokenizers are optimized for transformer models")
print("  - Subword tokenization handles out-of-vocabulary words better")
print("  - Each model has its own tokenizer (trained together)")
print("  - Documentation: https://huggingface.co/docs/transformers/preprocessing")


---

## 5. Complete Search System with Industry Tools

Let's build a complete search system combining all the tools:


In [ ]:
class MovieSearchSystem:
    """
    Complete movie search system using industry tools.
    
    Combines:
    - spaCy for preprocessing
    - scikit-learn for TF-IDF
    - scikit-learn for similarity search
    """
    
    def __init__(self, documents, use_spacy=True):
        """
        Initialize the search system.
        
        Args:
            documents: List of document texts
            use_spacy: Whether to use spaCy for preprocessing (faster, better)
        """
        self.documents = documents
        self.use_spacy = use_spacy and nlp is not None
        
        # Create tokenizer
        if self.use_spacy:
            def tokenizer(text):
                doc = nlp(text.lower())
                return [token.lemma_ for token in doc 
                        if not token.is_stop and not token.is_punct and not token.is_space]
        else:
            def tokenizer(text):
                tokens = word_tokenize(text.lower())
                stop_words = set(stopwords.words('english'))
                tokens = [token for token in tokens if token.isalnum() and token not in stop_words]
                lemmatizer = WordNetLemmatizer()
                return [lemmatizer.lemmatize(token) for token in tokens]
        
        # Create vectorizer
        self.vectorizer = TfidfVectorizer(
            tokenizer=tokenizer,
            max_features=200,
            lowercase=False
        )
        
        # Fit and transform documents
        self.vectors = self.vectorizer.fit_transform(documents)
        print(f"✓ Search system initialized: {self.vectors.shape}")
    
    def search(self, query, top_k=5):
        """
        Search for similar documents.
        
        Args:
            query: Search query string
            top_k: Number of results to return
        
        Returns:
            List of (index, similarity_score) tuples
        """
        # Convert query to vector
        query_vector = self.vectorizer.transform([query])
        
        # Calculate similarities
        similarities = cosine_similarity(query_vector, self.vectors)[0]
        
        # Get top_k results
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = [(idx, similarities[idx]) for idx in top_indices]
        return results

# Create search system
print("=" * 70)
print("Complete Search System with Industry Tools")
print("=" * 70)

if nlp:
    search_system = MovieSearchSystem(df['description'].tolist(), use_spacy=True)
    
    # Test queries
    test_queries = [
        "space adventure",
        "romantic comedy",
        "action thriller"
    ]
    
    for query in test_queries:
        print(f"\n{'='*70}")
        print(f"Query: '{query}'")
        print(f"{'='*70}")
        
        results = search_system.search(query, top_k=3)
        
        for i, (idx, score) in enumerate(results, 1):
            print(f"\n{i}. {df.iloc[idx]['title']} (similarity: {score:.3f})")
            print(f"   {df.iloc[idx]['description'][:80]}...")
    
    print("\n💡 This search system uses:")
    print("  - spaCy for advanced preprocessing (lemmatization, POS tagging)")
    print("  - scikit-learn for TF-IDF vectorization")
    print("  - scikit-learn for cosine similarity")
    print("  - All industry-standard, production-ready tools!")
else:
    print("⚠️  spaCy not available. System would use NLTK instead.")


---

## 6. Performance Comparison

Let's compare the performance of different approaches:


In [ ]:
import time

# Sample documents for testing
test_docs = df['description'].head(100).tolist()

print("=" * 70)
print("Performance Comparison")
print("=" * 70)

# Method 1: Basic scikit-learn (built-in tokenizer)
print("\n1. Basic scikit-learn (built-in):")
start = time.time()
basic_vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
basic_vectors = basic_vectorizer.fit_transform(test_docs)
basic_time = time.time() - start
print(f"   Time: {basic_time:.3f} seconds")
print(f"   Shape: {basic_vectors.shape}")

# Method 2: NLTK tokenizer
print("\n2. NLTK tokenizer:")
def nltk_tokenizer(text):
    tokens = word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token.isalnum() and token not in stop_words]
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]

start = time.time()
nltk_vectorizer = TfidfVectorizer(tokenizer=nltk_tokenizer, max_features=100, lowercase=False)
nltk_vectors = nltk_vectorizer.fit_transform(test_docs)
nltk_time = time.time() - start
print(f"   Time: {nltk_time:.3f} seconds")
print(f"   Shape: {nltk_vectors.shape}")

# Method 3: spaCy tokenizer
if nlp:
    print("\n3. spaCy tokenizer:")
    def spacy_tokenizer(text):
        doc = nlp(text.lower())
        return [token.lemma_ for token in doc 
                if not token.is_stop and not token.is_punct and not token.is_space]
    
    start = time.time()
    spacy_vectorizer = TfidfVectorizer(tokenizer=spacy_tokenizer, max_features=100, lowercase=False)
    spacy_vectors = spacy_vectorizer.fit_transform(test_docs)
    spacy_time = time.time() - start
    print(f"   Time: {spacy_time:.3f} seconds")
    print(f"   Shape: {spacy_vectors.shape}")
else:
    spacy_time = None
    print("\n3. spaCy tokenizer: ⚠️  Not available")

print("\n" + "=" * 70)
print("Summary:")
print("=" * 70)
print(f"Basic scikit-learn: {basic_time:.3f}s")
print(f"NLTK: {nltk_time:.3f}s ({nltk_time/basic_time:.1f}x slower)")
if spacy_time:
    print(f"spaCy: {spacy_time:.3f}s ({spacy_time/basic_time:.1f}x slower)")
    print("\n💡 Note: spaCy is optimized with Cython, so it's faster than NLTK")
    print("   For production systems, spaCy is often the best choice!")
    print("   Documentation: https://spacy.io/usage/facts-figures#speed")


---

## 7. When to Use Which Tool?

**Decision Guide:**

| Task | Recommended Tool | Why | Documentation |
|------|-----------------|-----|---------------|
| **Quick prototyping** | NLTK | Easy to use, lots of examples | https://www.nltk.org/ |
| **Production NLP** | spaCy | Fast, accurate, well-maintained | https://spacy.io/ |
| **Transformer models** | Hugging Face | Industry standard for ML models | https://huggingface.co/docs/transformers |
| **TF-IDF / ML** | scikit-learn | Best for traditional ML tasks | https://scikit-learn.org/ |
| **Research / Learning** | NLTK | Great documentation, educational | https://www.nltk.org/book/ |

**💡 Pro Tips:**
1. **Start with spaCy** for most production tasks - it's fast and accurate
2. **Use NLTK** when you need specific linguistic features or are learning
3. **Use Hugging Face** when working with transformer models (BERT, GPT, etc.)
4. **Combine tools** - use spaCy for preprocessing, scikit-learn for ML
5. **Always check documentation** - these tools are constantly updated!

**📚 Key Documentation Links:**
- **NLTK Book**: https://www.nltk.org/book/ (Great for learning!)
- **spaCy Usage Guide**: https://spacy.io/usage/spacy-101
- **Hugging Face Course**: https://huggingface.co/course/
- **scikit-learn User Guide**: https://scikit-learn.org/stable/user_guide.html


---

## Summary

**What you learned in Part 3:**

✅ **NLTK**: Natural Language Toolkit for preprocessing and tokenization
- Tokenization, stop words, stemming, lemmatization
- Great for learning and quick prototyping
- Documentation: https://www.nltk.org/

✅ **spaCy**: Industrial-strength NLP library
- Fast, accurate preprocessing with built-in models
- Best for production systems
- Documentation: https://spacy.io/

✅ **Hugging Face**: Modern transformers and tokenizers
- Subword tokenization for transformer models
- Industry standard for ML/NLP models
- Documentation: https://huggingface.co/docs/transformers

✅ **scikit-learn**: Machine learning and feature extraction
- TF-IDF vectorization with custom tokenizers
- Similarity search and ML algorithms
- Documentation: https://scikit-learn.org/

**Key Takeaways:**
- Industry tools make development faster and more reliable
- Understanding fundamentals (Parts 1 & 2) helps you use tools effectively
- Different tools excel at different tasks - choose wisely!
- Always refer to official documentation for the latest features

**Next Steps:**
- Practice with the **Exercise Notebook Part 3**
- Explore the documentation links provided
- Try combining different tools for your projects
- Continue to Class 3 for embeddings and semantic search!

**📚 Remember**: Documentation is your friend! Bookmark these links:
- NLTK: https://www.nltk.org/
- spaCy: https://spacy.io/
- Hugging Face: https://huggingface.co/docs/transformers
- scikit-learn: https://scikit-learn.org/
